In [ ]:
import pandas as pd
import os
import re

In [ ]:
df_raw = pd.read_csv('data/raw/psu_survey_raw.csv', encoding='utf-8-sig')

print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")
df_raw.head()

In [ ]:
# 1. Create a working copy so df_raw stays untouched
df_clean = df_raw.copy()

# 2. Fix missing value in q44_likelihood_use_loan (row index 101)
df_clean.loc[101, 'q44_likelihood_use_loan'] = 'I do not use any money lending/loaning service'

# 3. Select all text columns cleanly (silences Pandas4Warning)
text_cols = df_clean.select_dtypes(include=['object', 'str']).columns

# 4. Normalize non-ASCII symbols and strip extra whitespace across all text columns
char_replacements = {
    '’': "'",
    '–': '-',
    '₱': 'PHP '
}

for col in text_cols:
    for old_char, new_char in char_replacements.items():
        df_clean[col] = df_clean[col].str.replace(old_char, new_char, regex=False)
    df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True).str.strip()

# 5. Final verification check
remaining_non_ascii = set()
for col in text_cols:
    for val in df_clean[col].dropna().astype(str):
        remaining_non_ascii.update(re.findall(r'[^\x00-\x7F]', val))

print(f"Missing values: {df_clean.isnull().sum().sum()}")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")
print(f"Remaining non-ASCII: {remaining_non_ascii}")

# 6. Export cleaned dataset
output_dir = 'data/processed'
output_path = os.path.join(output_dir, 'psu_financial_literacy_cleaned.csv')
os.makedirs(output_dir, exist_ok=True)
df_clean.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")